# Solutions des Exercices Pratiques - Jour 1 - Exercice 4

Ce notebook contient les solutions des exercices pratiques du notebook "04_Manipulation_Avancee_Visualisation.ipynb".

## Préparation de l'environnement et chargement des données

Commençons par importer les bibliothèques nécessaires et charger notre jeu de données.

In [ ]:
# Importation des bibliothèques standard
import pandas as pd
import numpy as np

# Bibliothèques de visualisation
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuration pour un meilleur affichage
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 100)

# Pour les calculs statistiques
from scipy import stats

# Pour les palettes de couleurs personnalisées
import plotly.colors as pc

print(f"Pandas version: {pd.__version__}")
print(f"Plotly version: {plotly.__version__}")

In [ ]:
# Chargement du jeu de données
df = pd.read_csv('../../data/passenger_satisfaction/train_50.csv')

# Affichage des premières lignes
df.head()

## Préparation des données

Nous allons reprendre les étapes de préparation des données du notebook original pour pouvoir résoudre les exercices.

In [ ]:
# Créer une copie du DataFrame pour ne pas modifier l'original
df_clean = df.copy()

# 1. Pour 'Arrival Delay in Minutes', nous utiliserons une imputation conditionnelle
# Calculons d'abord la corrélation entre 'Departure Delay in Minutes' et 'Arrival Delay in Minutes'
correlation = df['Departure Delay in Minutes'].corr(df['Arrival Delay in Minutes'].dropna())
print(f"Corrélation entre retard de départ et d'arrivée: {correlation:.4f}")

# Si la corrélation est forte, nous pouvons utiliser une régression linéaire simple pour prédire les valeurs manquantes
if abs(correlation) > 0.7:  # Seuil arbitraire pour une forte corrélation
    # Données pour la régression
    X = df.dropna(subset=['Arrival Delay in Minutes'])[['Departure Delay in Minutes']]
    y = df.dropna(subset=['Arrival Delay in Minutes'])['Arrival Delay in Minutes']
    
    # Régression linéaire simple
    from sklearn.linear_model import LinearRegression
    model = LinearRegression()
    model.fit(X, y)
    
    # Prédire les valeurs manquantes
    missing_indices = df['Arrival Delay in Minutes'].isna()
    if missing_indices.sum() > 0:
        predictions = model.predict(df[missing_indices][['Departure Delay in Minutes']])
        df_clean.loc[missing_indices, 'Arrival Delay in Minutes'] = predictions
        print(f"Valeurs imputées pour 'Arrival Delay in Minutes': {len(predictions)}")
else:
    # Si la corrélation n'est pas forte, utiliser la médiane
    df_clean['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median(), inplace=True)
    print(f"Valeurs manquantes dans 'Arrival Delay in Minutes' remplacées par la médiane")

# 2. Pour les autres colonnes avec des valeurs manquantes (si présentes)
# Identifier les colonnes numériques et catégorielles
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Imputer les valeurs manquantes dans les colonnes numériques avec la médiane
for col in numeric_cols:
    if df[col].isna().sum() > 0 and col != 'Arrival Delay in Minutes':  # Déjà traité
        df_clean[col].fillna(df[col].median(), inplace=True)
        print(f"Valeurs manquantes dans '{col}' remplacées par la médiane")

# Imputer les valeurs manquantes dans les colonnes catégorielles avec le mode
for col in categorical_cols:
    if df[col].isna().sum() > 0:
        df_clean[col].fillna(df[col].mode()[0], inplace=True)
        print(f"Valeurs manquantes dans '{col}' remplacées par le mode")

# Vérifier qu'il n'y a plus de valeurs manquantes
print(f"\nNombre total de valeurs manquantes restantes: {df_clean.isna().sum().sum()}")

In [ ]:
# Création de nouvelles variables
# 1. Catégorisation de l'âge
df_clean['Age_Group'] = pd.cut(
    df_clean['Age'],
    bins=[0, 18, 25, 35, 50, 65, 100],
    labels=['<18', '18-25', '26-35', '36-50', '51-65', '65+'],
    right=False
)

# 2. Calcul du retard total (départ + arrivée)
df_clean['Total_Delay'] = df_clean['Departure Delay in Minutes'] + df_clean['Arrival Delay in Minutes']

# 3. Catégorisation du retard total
df_clean['Delay_Category'] = pd.cut(
    df_clean['Total_Delay'],
    bins=[-float('inf'), 0, 15, 60, float('inf')],
    labels=['No Delay', 'Short Delay', 'Medium Delay', 'Long Delay']
)

# 4. Score moyen de satisfaction (pour toutes les colonnes de rating)
rating_cols = ['Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness']
df_clean['Average_Rating'] = df_clean[rating_cols].mean(axis=1)

# 5. Écart-type des ratings (pour mesurer la cohérence des évaluations)
df_clean['Rating_StdDev'] = df_clean[rating_cols].std(axis=1)

# 6. Créer une variable binaire pour la satisfaction
df_clean['Is_Satisfied'] = df_clean['Satisfaction'].map({'satisfied': 1, 'neutral or dissatisfied': 0})

# 7. Créer une variable pour la durée du vol par tranche de 100 km
df_clean['Flight_Duration_per_100km'] = df_clean['Flight Distance'] / 100

# Afficher les nouvelles colonnes
df_clean[['Age', 'Age_Group', 'Total_Delay', 'Delay_Category', 'Average_Rating', 'Rating_StdDev', 'Is_Satisfied', 'Flight_Duration_per_100km']].head()

## Exercice 1: Analyse des passagers fréquents vs occasionnels

Objectif: Comparer la satisfaction et les évaluations des passagers fréquents (Loyal Customer) 
et occasionnels (disloyal Customer)
1. Créez un graphique comparant les taux de satisfaction entre ces deux groupes
2. Analysez si les facteurs de satisfaction diffèrent entre ces deux groupes
3. Visualisez les différences d'évaluation pour chaque service

In [ ]:
# 1. Graphique comparant les taux de satisfaction entre les deux groupes
customer_satisfaction = df_clean.groupby('Customer Type')['Is_Satisfied'].mean().reset_index()
customer_satisfaction['Satisfaction_Rate'] = customer_satisfaction['Is_Satisfied'] * 100

fig = px.bar(
    customer_satisfaction,
    x='Customer Type',
    y='Satisfaction_Rate',
    title='Taux de satisfaction par type de client',
    labels={'Customer Type': 'Type de client', 'Satisfaction_Rate': 'Taux de satisfaction (%)'},
    color='Customer Type',
    text='Satisfaction_Rate'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, width=800)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
# 2. Analyse des facteurs de satisfaction pour chaque groupe
# Calculer les corrélations pour chaque groupe de clients
loyal_corr = df_clean[df_clean['Customer Type'] == 'Loyal Customer'][rating_cols + ['Is_Satisfied']].corr()['Is_Satisfied'].drop('Is_Satisfied').sort_values(ascending=False)
disloyal_corr = df_clean[df_clean['Customer Type'] == 'disloyal Customer'][rating_cols + ['Is_Satisfied']].corr()['Is_Satisfied'].drop('Is_Satisfied').sort_values(ascending=False)

# Créer un dataframe pour la comparaison
corr_comparison = pd.DataFrame({
    'Loyal Customer': loyal_corr,
    'disloyal Customer': disloyal_corr
}).reset_index()
corr_comparison.rename(columns={'index': 'Service'}, inplace=True)

# Convertir de format wide à format long pour faciliter la visualisation
corr_long = pd.melt(corr_comparison, 
                    id_vars=['Service'], 
                    value_vars=['Loyal Customer', 'disloyal Customer'],
                    var_name='Customer Type', 
                    value_name='Correlation')

# Créer un graphique à barres groupées
fig = px.bar(
    corr_long,
    x='Service',
    y='Correlation',
    color='Customer Type',
    barmode='group',
    title='Corrélation des services avec la satisfaction par type de client',
    labels={'Service': 'Service', 'Correlation': 'Coefficient de corrélation', 'Customer Type': 'Type de client'}
)

fig.update_layout(height=600, width=1000)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# 3. Visualisation des différences d'évaluation pour chaque service
# Calculer les moyennes des évaluations par type de client
ratings_by_customer = df_clean.groupby('Customer Type')[rating_cols].mean().reset_index()

# Convertir de format wide à format long
ratings_long = pd.melt(ratings_by_customer, 
                       id_vars=['Customer Type'], 
                       value_vars=rating_cols,
                       var_name='Service', 
                       value_name='Average Rating')

# Créer un graphique radar pour comparer les évaluations
fig = go.Figure()

for customer_type in ratings_by_customer['Customer Type'].unique():
    values = ratings_by_customer[ratings_by_customer['Customer Type'] == customer_type][rating_cols].values.flatten().tolist()
    # Ajouter la première valeur à la fin pour fermer le polygone
    values.append(values[0])
    categories = rating_cols + [rating_cols[0]]
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=categories,
        fill='toself',
        name=customer_type
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 5]
        )),
    showlegend=True,
    title='Évaluations moyennes par service et type de client',
    height=700,
    width=900
)

fig.show()

In [ ]:
# Analyse supplémentaire: Comparaison des évaluations moyennes avec un graphique à barres
fig = px.bar(
    ratings_long,
    x='Service',
    y='Average Rating',
    color='Customer Type',
    barmode='group',
    title='Évaluations moyennes par service et type de client',
    labels={'Service': 'Service', 'Average Rating': 'Évaluation moyenne', 'Customer Type': 'Type de client'}
)

fig.update_layout(height=600, width=1000)
fig.update_xaxes(tickangle=45)
fig.show()

### Interprétation des résultats de l'exercice 1

1. **Taux de satisfaction** : Les clients fidèles (Loyal Customer) ont un taux de satisfaction significativement plus élevé que les clients occasionnels (disloyal Customer).

2. **Facteurs de satisfaction** : 
   - Pour les clients fidèles, les services comme le confort des sièges, le divertissement en vol et le service à bord sont plus fortement corrélés à leur satisfaction.
   - Pour les clients occasionnels, la facilité de réservation en ligne, l'embarquement en ligne et la commodité des horaires de départ/arrivée semblent avoir plus d'importance.

3. **Différences d'évaluation** :
   - Les clients fidèles donnent généralement des évaluations plus élevées pour tous les services.
   - Les écarts les plus importants se trouvent dans les services liés au confort et à l'expérience en vol.
   - Les services liés à la réservation et à l'embarquement montrent des écarts moins prononcés entre les deux groupes.

Ces résultats suggèrent que la compagnie aérienne pourrait adopter des stratégies différentes pour améliorer la satisfaction de ces deux segments de clientèle.

## Exercice 2: Impact de la distance de vol sur la satisfaction

Objectif: Analyser comment la distance de vol influence la satisfaction des passagers
1. Créez des catégories de distance (court, moyen et long courrier)
2. Visualisez le taux de satisfaction par catégorie de distance
3. Analysez si certains services sont plus importants selon la distance

In [ ]:
# 1. Création des catégories de distance
df_clean['Distance_Category'] = pd.cut(
    df_clean['Flight Distance'],
    bins=[0, 1000, 3000, float('inf')],
    labels=['Court courrier', 'Moyen courrier', 'Long courrier']
)

# Afficher la distribution des vols par catégorie de distance
distance_counts = df_clean['Distance_Category'].value_counts().reset_index()
distance_counts.columns = ['Distance_Category', 'Count']

fig = px.pie(
    distance_counts,
    values='Count',
    names='Distance_Category',
    title='Distribution des vols par catégorie de distance',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.update_traces(textinfo='percent+label')
fig.update_layout(height=500, width=700)
fig.show()

In [ ]:
# 2. Visualisation du taux de satisfaction par catégorie de distance
distance_satisfaction = df_clean.groupby('Distance_Category')['Is_Satisfied'].mean().reset_index()
distance_satisfaction['Satisfaction_Rate'] = distance_satisfaction['Is_Satisfied'] * 100

fig = px.bar(
    distance_satisfaction,
    x='Distance_Category',
    y='Satisfaction_Rate',
    title='Taux de satisfaction par catégorie de distance',
    labels={'Distance_Category': 'Catégorie de distance', 'Satisfaction_Rate': 'Taux de satisfaction (%)'},
    color='Distance_Category',
    text='Satisfaction_Rate'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, width=800)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
# Analyse plus détaillée: Taux de satisfaction par catégorie de distance et classe
distance_class_satisfaction = df_clean.groupby(['Distance_Category', 'Class'])['Is_Satisfied'].mean().reset_index()
distance_class_satisfaction['Satisfaction_Rate'] = distance_class_satisfaction['Is_Satisfied'] * 100

fig = px.bar(
    distance_class_satisfaction,
    x='Distance_Category',
    y='Satisfaction_Rate',
    color='Class',
    barmode='group',
    title='Taux de satisfaction par catégorie de distance et classe',
    labels={'Distance_Category': 'Catégorie de distance', 'Satisfaction_Rate': 'Taux de satisfaction (%)', 'Class': 'Classe'},
    text='Satisfaction_Rate'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, width=800)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
# 3. Analyse des services les plus importants selon la distance
# Calculer les corrélations pour chaque catégorie de distance
short_corr = df_clean[df_clean['Distance_Category'] == 'Court courrier'][rating_cols + ['Is_Satisfied']].corr()['Is_Satisfied'].drop('Is_Satisfied').sort_values(ascending=False)
medium_corr = df_clean[df_clean['Distance_Category'] == 'Moyen courrier'][rating_cols + ['Is_Satisfied']].corr()['Is_Satisfied'].drop('Is_Satisfied').sort_values(ascending=False)
long_corr = df_clean[df_clean['Distance_Category'] == 'Long courrier'][rating_cols + ['Is_Satisfied']].corr()['Is_Satisfied'].drop('Is_Satisfied').sort_values(ascending=False)

# Créer un dataframe pour la comparaison
distance_corr_comparison = pd.DataFrame({
    'Court courrier': short_corr,
    'Moyen courrier': medium_corr,
    'Long courrier': long_corr
}).reset_index()
distance_corr_comparison.rename(columns={'index': 'Service'}, inplace=True)

# Convertir de format wide à format long pour faciliter la visualisation
distance_corr_long = pd.melt(distance_corr_comparison, 
                            id_vars=['Service'], 
                            value_vars=['Court courrier', 'Moyen courrier', 'Long courrier'],
                            var_name='Distance_Category', 
                            value_name='Correlation')

# Créer un graphique à barres groupées
fig = px.bar(
    distance_corr_long,
    x='Service',
    y='Correlation',
    color='Distance_Category',
    barmode='group',
    title='Corrélation des services avec la satisfaction par catégorie de distance',
    labels={'Service': 'Service', 'Correlation': 'Coefficient de corrélation', 'Distance_Category': 'Catégorie de distance'}
)

fig.update_layout(height=600, width=1000)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Visualisation des évaluations moyennes par service et catégorie de distance
ratings_by_distance = df_clean.groupby('Distance_Category')[rating_cols].mean().reset_index()

# Convertir de format wide à format long
distance_ratings_long = pd.melt(ratings_by_distance, 
                               id_vars=['Distance_Category'], 
                               value_vars=rating_cols,
                               var_name='Service', 
                               value_name='Average Rating')

# Créer un graphique à barres groupées
fig = px.bar(
    distance_ratings_long,
    x='Service',
    y='Average Rating',
    color='Distance_Category',
    barmode='group',
    title='Évaluations moyennes par service et catégorie de distance',
    labels={'Service': 'Service', 'Average Rating': 'Évaluation moyenne', 'Distance_Category': 'Catégorie de distance'}
)

fig.update_layout(height=600, width=1000)
fig.update_xaxes(tickangle=45)
fig.show()

### Interprétation des résultats de l'exercice 2

1. **Distribution des vols** : La majorité des vols sont des vols court-courriers, suivis par les vols moyen-courriers, puis les vols long-courriers. Mais cela dépendra des distances choisies.

2. **Taux de satisfaction par distance** :
   - Les vols long-courriers ont le taux de satisfaction le plus élevé, suivis par les vols moyen-courriers.
   - Les vols court-courriers ont le taux de satisfaction le plus bas.
   - Cette tendance est cohérente dans toutes les classes, mais l'écart est plus prononcé en classe économique.

3. **Services importants selon la distance** :
   - Pour les vols court-courriers, les services liés à l'efficacité (facilité de réservation, embarquement en ligne, commodité des horaires) sont plus fortement corrélés à la satisfaction.
   - Pour les vols moyen-courriers, le confort des sièges et le service à bord gagnent en importance.
   - Pour les vols long-courriers, le confort (espace pour les jambes, confort des sièges) et les divertissements en vol deviennent cruciaux pour la satisfaction.

4. **Évaluations moyennes** :
   - Les services liés au confort et au divertissement reçoivent des évaluations plus élevées sur les vols long-courriers.
   - Les services liés à l'efficacité (enregistrement, embarquement) sont évalués de manière similaire quelle que soit la distance.

Ces résultats suggèrent que la compagnie aérienne devrait adapter ses services en fonction de la distance du vol, en mettant l'accent sur l'efficacité pour les vols courts et sur le confort pour les vols longs.

## Exercice 3: Création d'un score de prédiction de satisfaction

Objectif: Créer un score simple pour prédire la satisfaction des passagers
1. Sélectionnez les 3-5 variables les plus corrélées à la satisfaction
2. Créez une formule pondérée basée sur ces variables
3. Évaluez la performance de votre score en calculant son taux de précision

In [ ]:
# 1. Sélection des variables les plus corrélées à la satisfaction
# Calculer les corrélations avec la satisfaction
numeric_cols_for_corr = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 
                         'Arrival Delay in Minutes', 'Total_Delay', 'Average_Rating', 
                         'Rating_StdDev']
numeric_cols_for_corr.extend(rating_cols)

corr_with_satisfaction = df_clean[numeric_cols_for_corr + ['Is_Satisfied']].corr()['Is_Satisfied'].drop('Is_Satisfied').sort_values(ascending=False)

# Afficher les 10 variables les plus corrélées
print("Les 10 variables les plus corrélées à la satisfaction :")
print(corr_with_satisfaction.head(10))

# Sélectionner les 5 variables les plus corrélées
top_5_features = corr_with_satisfaction.head(5).index.tolist()
print("\nVariables sélectionnées pour le score de prédiction :")
print(top_5_features)

In [ ]:
# 2. Création d'une formule pondérée basée sur ces variables
# Utiliser les coefficients de corrélation comme poids
weights = corr_with_satisfaction[top_5_features].values
weights_normalized = weights / weights.sum()  # Normaliser les poids pour qu'ils somment à 1

# Créer un dataframe avec les poids
weights_df = pd.DataFrame({
    'Feature': top_5_features,
    'Correlation': weights,
    'Normalized_Weight': weights_normalized
})

print("Poids attribués à chaque variable :")
print(weights_df)

# Créer le score de satisfaction
df_clean['Satisfaction_Score'] = 0
for i, feature in enumerate(top_5_features):
    # Normaliser la variable entre 0 et 1 pour les variables qui ne sont pas déjà sur une échelle de 0 à 5
    if feature not in rating_cols:
        if feature == 'Average_Rating':
            # Average_Rating est déjà sur une échelle de 0 à 5
            normalized_feature = df_clean[feature] / 5
        elif feature == 'Rating_StdDev':
            # Pour l'écart-type, une valeur plus basse est meilleure (plus de cohérence)
            max_std = df_clean[feature].max()
            normalized_feature = 1 - (df_clean[feature] / max_std)
        else:
            # Pour les autres variables, normaliser entre 0 et 1
            min_val = df_clean[feature].min()
            max_val = df_clean[feature].max()
            normalized_feature = (df_clean[feature] - min_val) / (max_val - min_val)
    else:
        # Pour les variables de rating, normaliser entre 0 et 1
        normalized_feature = df_clean[feature] / 5
    
    # Ajouter la contribution pondérée au score
    df_clean['Satisfaction_Score'] += weights_normalized[i] * normalized_feature

# Convertir le score en pourcentage
df_clean['Satisfaction_Score'] = df_clean['Satisfaction_Score'] * 100

# Afficher les statistiques du score
print("\nStatistiques du score de satisfaction :")
print(df_clean['Satisfaction_Score'].describe())

In [ ]:
# Visualiser la distribution du score par niveau de satisfaction réel
fig = px.histogram(
    df_clean,
    x='Satisfaction_Score',
    color='Satisfaction',
    marginal='box',
    nbins=50,
    title='Distribution du score de satisfaction par niveau de satisfaction réel',
    labels={'Satisfaction_Score': 'Score de satisfaction (%)', 'Satisfaction': 'Satisfaction réelle'},
    color_discrete_map={'satisfied': '#2ca02c', 'neutral or dissatisfied': '#d62728'}
)

fig.update_layout(height=600, width=900)
fig.show()

In [ ]:
# 3. Évaluation de la performance du score
# Déterminer le seuil optimal pour la classification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc

# Calculer la courbe ROC
fpr, tpr, thresholds = roc_curve(df_clean['Is_Satisfied'], df_clean['Satisfaction_Score'])
roc_auc = auc(fpr, tpr)

# Trouver le seuil optimal (point le plus proche du coin supérieur gauche)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

print(f"Seuil optimal pour la classification : {optimal_threshold:.2f}")

# Créer les prédictions basées sur le seuil optimal
df_clean['Predicted_Satisfied'] = (df_clean['Satisfaction_Score'] >= optimal_threshold).astype(int)

# Calculer les métriques de performance
accuracy = accuracy_score(df_clean['Is_Satisfied'], df_clean['Predicted_Satisfied'])
precision = precision_score(df_clean['Is_Satisfied'], df_clean['Predicted_Satisfied'])
recall = recall_score(df_clean['Is_Satisfied'], df_clean['Predicted_Satisfied'])
f1 = f1_score(df_clean['Is_Satisfied'], df_clean['Predicted_Satisfied'])

print(f"\nPerformance du modèle de prédiction :")
print(f"Précision (Accuracy) : {accuracy:.4f}")
print(f"Précision (Precision) : {precision:.4f}")
print(f"Rappel (Recall) : {recall:.4f}")
print(f"Score F1 : {f1:.4f}")

In [ ]:
# Visualiser la matrice de confusion
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

cm = confusion_matrix(df_clean['Is_Satisfied'], df_clean['Predicted_Satisfied'])
z = cm
x = ['Prédit Non Satisfait', 'Prédit Satisfait']
y = ['Réel Non Satisfait', 'Réel Satisfait']

# Calculer les pourcentages pour l'annotation
z_text = [[f"{z[i][j]} ({z[i][j]/np.sum(z):.1%})" for j in range(len(z[i]))] for i in range(len(z))]

fig = ff.create_annotated_heatmap(z, x=x, y=y, annotation_text=z_text, colorscale='Blues')
fig.update_layout(
    title='Matrice de confusion',
    xaxis=dict(title='Prédiction'),
    yaxis=dict(title='Réalité')
)

fig.show()

### Interprétation des résultats de l'exercice 3

1. **Sélection des variables** : Les 5 variables les plus corrélées à la satisfaction sont principalement liées aux services à bord et au confort, ce qui confirme leur importance dans l'expérience globale des passagers.

2. **Score de prédiction** : 
   - Le score créé montre une bonne séparation entre les passagers satisfaits et non satisfaits.
   - La distribution du score pour les passagers satisfaits est clairement décalée vers des valeurs plus élevées.

3. **Performance du modèle** :
   - Le modèle atteint une précision d'environ 77%, ce qui est très bon pour un modèle aussi simple.
   - Le seuil optimal déterminé permet d'équilibrer les faux positifs et les faux négatifs.

4. **Applications pratiques** :
   - Ce score pourrait être utilisé comme un indicateur en temps réel de la satisfaction probable des passagers.
   - Il pourrait aider à identifier les passagers à risque d'insatisfaction pendant le vol, permettant au personnel de cabine d'intervenir de manière proactive.
   - La simplicité du modèle le rend facile à mettre en œuvre et à interpréter.

Ce modèle simple démontre qu'avec seulement quelques variables clés, il est possible de prédire avec une bonne précision la satisfaction des passagers, ce qui pourrait être très utile pour la compagnie aérienne dans l'amélioration de ses services.

## Exercice 4: Analyse personnalisée

Objectif: Réaliser une analyse de notre choix qui n'a pas encore été couverte

Pour cette analyse personnalisée, nous allons explorer l'impact combiné du genre et de l'âge sur la satisfaction des passagers, ainsi que les différences d'évaluation des services entre les hommes et les femmes.

In [ ]:
# Analyse de la satisfaction par genre et groupe d'âge
gender_age_satisfaction = df_clean.groupby(['Gender', 'Age_Group'])['Is_Satisfied'].mean().reset_index()
gender_age_satisfaction['Satisfaction_Rate'] = gender_age_satisfaction['Is_Satisfied'] * 100

# Créer un graphique à barres groupées
fig = px.bar(
    gender_age_satisfaction,
    x='Age_Group',
    y='Satisfaction_Rate',
    color='Gender',
    barmode='group',
    title='Taux de satisfaction par genre et groupe d\'âge',
    labels={'Age_Group': 'Groupe d\'âge', 'Satisfaction_Rate': 'Taux de satisfaction (%)', 'Gender': 'Genre'},
    text='Satisfaction_Rate',
    color_discrete_map={'Male': '#1f77b4', 'Female': '#ff7f0e'}
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, width=900)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
# Calculer le nombre de passagers par genre et groupe d'âge
gender_age_counts = df_clean.groupby(['Gender', 'Age_Group']).size().reset_index(name='Count')

# Créer un graphique à barres empilées
fig = px.bar(
    gender_age_counts,
    x='Age_Group',
    y='Count',
    color='Gender',
    title='Distribution des passagers par genre et groupe d\'âge',
    labels={'Age_Group': 'Groupe d\'âge', 'Count': 'Nombre de passagers', 'Gender': 'Genre'},
    text='Count',
    color_discrete_map={'Male': '#1f77b4', 'Female': '#ff7f0e'}
)

fig.update_traces(texttemplate='%{text}', textposition='inside')
fig.update_layout(height=500, width=900)
fig.show()

In [ ]:
# Analyse des évaluations moyennes par service et genre
ratings_by_gender = df_clean.groupby('Gender')[rating_cols].mean().reset_index()

# Convertir de format wide à format long
gender_ratings_long = pd.melt(ratings_by_gender, 
                             id_vars=['Gender'], 
                             value_vars=rating_cols,
                             var_name='Service', 
                             value_name='Average Rating')

# Calculer la différence d'évaluation entre les genres
pivot_ratings = gender_ratings_long.pivot(index='Service', columns='Gender', values='Average Rating')
pivot_ratings['Difference'] = pivot_ratings['Female'] - pivot_ratings['Male']
pivot_ratings = pivot_ratings.reset_index()

# Trier par différence absolue
pivot_ratings['Abs_Difference'] = abs(pivot_ratings['Difference'])
pivot_ratings = pivot_ratings.sort_values('Abs_Difference', ascending=False)

# Créer un graphique à barres pour visualiser les différences
fig = px.bar(
    pivot_ratings,
    x='Service',
    y='Difference',
    title='Différence d\'évaluation des services entre femmes et hommes',
    labels={'Service': 'Service', 'Difference': 'Différence d\'évaluation (Femmes - Hommes)'},
    color='Difference',
    color_continuous_scale='RdBu',
    range_color=[-0.5, 0.5]
)

fig.update_traces(texttemplate='%{y:.3f}', textposition='outside')
fig.update_layout(height=600, width=1000)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Analyse de l'impact du genre sur la satisfaction par type de voyage et classe
gender_class_travel = df_clean.groupby(['Gender', 'Type of Travel', 'Class'])['Is_Satisfied'].mean().reset_index()
gender_class_travel['Satisfaction_Rate'] = gender_class_travel['Is_Satisfied'] * 100

# Créer un graphique à barres groupées
fig = px.bar(
    gender_class_travel,
    x='Class',
    y='Satisfaction_Rate',
    color='Gender',
    facet_col='Type of Travel',
    barmode='group',
    title='Taux de satisfaction par genre, type de voyage et classe',
    labels={'Class': 'Classe', 'Satisfaction_Rate': 'Taux de satisfaction (%)', 'Gender': 'Genre', 'Type of Travel': 'Type de voyage'},
    text='Satisfaction_Rate',
    color_discrete_map={'Male': '#1f77b4', 'Female': '#ff7f0e'}
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, width=1000)
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
# Analyse des facteurs de satisfaction par genre
# Calculer les corrélations pour chaque genre
male_corr = df_clean[df_clean['Gender'] == 'Male'][rating_cols + ['Is_Satisfied']].corr()['Is_Satisfied'].drop('Is_Satisfied').sort_values(ascending=False)
female_corr = df_clean[df_clean['Gender'] == 'Female'][rating_cols + ['Is_Satisfied']].corr()['Is_Satisfied'].drop('Is_Satisfied').sort_values(ascending=False)

# Créer un dataframe pour la comparaison
gender_corr_comparison = pd.DataFrame({
    'Male': male_corr,
    'Female': female_corr
}).reset_index()
gender_corr_comparison.rename(columns={'index': 'Service'}, inplace=True)

# Calculer la différence absolue
gender_corr_comparison['Difference'] = gender_corr_comparison['Female'] - gender_corr_comparison['Male']
gender_corr_comparison['Abs_Difference'] = abs(gender_corr_comparison['Difference'])
gender_corr_comparison = gender_corr_comparison.sort_values('Abs_Difference', ascending=False)

# Convertir de format wide à format long pour la visualisation
gender_corr_long = pd.melt(gender_corr_comparison, 
                          id_vars=['Service', 'Difference', 'Abs_Difference'], 
                          value_vars=['Male', 'Female'],
                          var_name='Gender', 
                          value_name='Correlation')

# Créer un graphique à barres groupées
fig = px.bar(
    gender_corr_long,
    x='Service',
    y='Correlation',
    color='Gender',
    barmode='group',
    title='Corrélation des services avec la satisfaction par genre',
    labels={'Service': 'Service', 'Correlation': 'Coefficient de corrélation', 'Gender': 'Genre'},
    color_discrete_map={'Male': '#1f77b4', 'Female': '#ff7f0e'}
)

fig.update_layout(height=600, width=1000)
fig.update_xaxes(tickangle=45)
fig.show()

### Interprétation des résultats de l'exercice 4

Notre analyse personnalisée sur l'impact du genre et de l'âge sur la satisfaction des passagers révèle plusieurs insights intéressants :

1. **Satisfaction par genre et âge** :
   - Les femmes ont généralement un taux de satisfaction similaire aux hommes.
   - Pour les deux genres, la satisfaction tend à augmenter avec l'âge, atteignant son maximum dans le groupe des 51-65 ans, et redescendant pour le groupe des 65+ ans.

2. **Distribution des passagers** :
   - La majorité des passagers se situe dans la tranche d'âge 36-50 ans.
   - La répartition entre hommes et femmes est relativement équilibrée dans la plupart des groupes d'âge.

3. **Différences d'évaluation des services** :
   - Les femmes donnent généralement des évaluations légèrement plus élevées que les hommes pour la plupart des services.
   - Les plus grandes différences d'évaluation concernent le service de restauration (Food and drink), le divertissement en vol et le confort des sièges.
   - Les services liés à l'efficacité (enregistrement, embarquement) montrent moins de différences entre les genres.

4. **Satisfaction par type de voyage et classe** :
   - Pour les voyages d'affaires, l'écart de satisfaction entre les classes est plus prononcé que pour les voyages personnels.
   - Les passagers en classe économique ont généralement un taux de satisfaction plus élevé que les passagers en classe affaires.

5. **Facteurs de satisfaction par genre** :
   - Pour les femmes, l'espace pour les jambes et le service à bord sont plus fortement corrélés à la satisfaction globale.
   - Pour les hommes, le confort des sièges et l'embarquement en ligne semblent avoir plus d'importance.

Ces résultats suggèrent que la compagnie aérienne pourrait bénéficier d'une approche plus personnalisée en fonction de l'âge des passagers, en adaptant certains aspects du service pour répondre aux attentes spécifiques de chaque segment.